# In the name of God
## HW6
### Practical Section: TRPO Algorithm


### PPO Algorithm






**Proximal Policy Optimization (PPO)** is a reinforcement learning (RL) algorithm designed to optimize policies in a way that balances ease of implementation, efficiency, and performance. It is a model-free, on-policy algorithm that is part of the family of policy gradient methods, like **REINFORCE** and **Actor-Critic** algorithms.

### **Key Concepts of PPO:**

1. **Policy Gradient Methods**:
   - In **policy gradient** methods, the goal is to directly optimize the policy by adjusting its parameters (usually through gradient-based methods) to maximize the expected return.
   - PPO is an **on-policy** method, meaning that the policy being optimized is the same as the one used to generate data (as opposed to off-policy methods like DQN, which use experiences from different policies).

2. **Objective**:
   - PPO’s primary goal is to **maximize the expected reward** from the environment by improving the policy over time. This is done by adjusting the policy parameters using **gradients** while ensuring the updates do not drastically change the policy, preventing performance collapse.
   
3. **Clipped Objective Function**:
   - A key idea in PPO is the use of a **clipped objective function**, which helps in maintaining a balance between exploration and exploitation. This clipped objective ensures that the policy updates are small enough to prevent large deviations from the current policy, which can be destabilizing.
   
   - The **clipped objective** is defined as:
     $$
     \mathbb{L}^{CLIP}(\theta) = \hat{\mathbb{E}}_t \left[ \min\left( r_t(\theta) \hat{A}_t, \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) \hat{A}_t \right) \right]
     $$
     where:
     - $ r_t(\theta) $ is the probability ratio: the ratio between the new policy probability and the old policy probability for a given action.
     - $ \hat{A}_t $ is the **advantage estimate**, which indicates how much better or worse the action performed compared to the average action.
     - $ \epsilon $ is a small hyperparameter (typically 0.1 or 0.2) that controls the extent of the clipping. The idea is to ensure that the policy does not deviate too much from the previous policy.

4. **Advantage Estimation**:
   - **Advantage estimation** (often using **Generalized Advantage Estimation (GAE)**) helps to reduce the variance in policy gradient methods. It is used to compute how much better a given action is compared to the average action in that state, which provides more stable updates.

5. **Multiple Epochs of Optimization**:
   - Unlike other policy gradient methods that update the policy after a single pass over the collected data, PPO updates the policy **over multiple epochs** using the same batch of data. This helps make more efficient use of each data sample.

6. **Generalized Advantage Estimation (GAE)**:
   - **GAE** is used to compute a more stable estimate of the advantage function by combining multiple time-step returns with a weight that decays over time. This helps strike a balance between bias and variance in the advantage function.

### **PPO Algorithm Steps**:

1. **Data Collection**:
   - Run the current policy in the environment to collect a batch of trajectories (state-action-reward sequences).
   
2. **Compute Advantages**:
   - Use the collected data to compute the **advantages** using methods like **GAE**.
   
3. **Policy Update (Clipped Surrogate Objective)**:
   - Update the policy using the **clipped objective function**, which ensures that the policy does not change too drastically.
   
4. **Repeat**:
   - Repeat the data collection and policy update steps for several iterations until convergence or the desired performance is achieved.

### **Key Advantages of PPO**:
1. **Stability**: The clipped objective function prevents large, unstable policy updates, making PPO more stable than some earlier methods like TRPO (Trust Region Policy Optimization).
   
2. **Simplicity**: PPO is relatively simple to implement and does not require complex optimization techniques like TRPO. It only requires the use of a standard **stochastic gradient descent (SGD)** optimizer.

3. **Efficient Use of Data**: PPO can be run over multiple epochs of the same batch of data, making it more sample-efficient compared to methods like DQN that only use a single pass over the data.

4. **Good Performance**: PPO has been shown to work well in many RL environments, including both discrete and continuous action spaces.

### **PPO vs. Other Algorithms**:
- **PPO vs. TRPO**: TRPO is another on-policy algorithm that guarantees a large improvement with each update. However, it is much more complex and requires more computational resources than PPO. PPO sacrifices some of the guarantees of TRPO for simplicity and computational efficiency.
- **PPO vs. DQN**: DQN is an off-policy algorithm that uses value-based methods and a Q-network to approximate the Q-values. PPO, being a policy-based method, directly optimizes the policy. This makes PPO suitable for environments with continuous action spaces, while DQN works better for discrete action spaces.

### **Conclusion**:
Proximal Policy Optimization (PPO) is a popular, stable, and computationally efficient reinforcement learning algorithm that uses a policy gradient method. It is highly effective in both discrete and continuous action spaces and has been widely used in both academic research and practical applications. Its key feature, the clipped objective function, ensures stable training by limiting large policy updates, making it one of the go-to algorithms in modern RL frameworks.

# Importing Required Libraries

First, we need to import the necessary libraries. We will be using OpenAI's `gym` for the Lunar Lander environment, `numpy` for numerical operations, and `torch` for implementing the neural network and optimization.


In [ ]:
!pip install --upgrade setuptools wheel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 67.7.2
    Uninstalling setuptools-67.7.2:
      Successfully uninstalled setuptools-67.7.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
lida 0.0.10 requires fastapi, which is not installed.
lida 0.0.10 requires kaleido, which is not installed.
lida 0.0.10 requires python-multipart, which is not installed.
lida 0.0.10 requires uvicorn, which is not installed.


In [ ]:
!pip install swig
!pip install gym[box2d]

  Using cached swig-4.1.1.post1-py2.py3-none-manylinux_2_5_x86_64.manylinux1_x86_64.whl (1.8 MB)
  Using cached box2d-py-2.3.5.tar.gz (374 kB)
  Preparing metadata (setup.py) ... done
  Using cached pygame-2.1.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Created wheel for box2d-py: filename=box2d_py-2.3.5-cp310-cp310-linux_x86_64.whl size=2373128 sha256=3b4b9332556f839483b763ee10224573920c89defdd92488dcfed82b6664066c
  Stored in directory: /root/.cache/pip/wheels/db/8f/6a/eaaadf056fba10a98d986f6dce954e6201ba3126926fc5ad9e
Successfully built box2d-py
  Attempting uninstall: pygame
    Found existing installation: pygame 2.5.2
    Uninstalling pygame-2.5.2:
      Successfully uninstalled pygame-2.5.2


In [ ]:
import gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Creating the Environment

We will create the Lunar Lander environment using the `gym.make()` function. We will also set the `enable_wind` parameter to `True` as mentioned.


In [ ]:
env = gym.make('LunarLander-v2')
env.enable_wind = True

/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


# Defining the Policy Network

We will define a simple policy network using PyTorch. This network will take the state of the environment as input and output the action probabilities and state value.


In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, num_inputs, num_actions, hidden_size, learning_rate=3e-4):
        super(PolicyNetwork, self).__init__()

        self.num_actions = num_actions
        self.linear1 = nn.Linear(num_inputs, hidden_size)
        self.linear2 = nn.Linear(hidden_size, num_actions)
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, state):
        x = torch.tanh(self.linear1(state))
        x = self.linear2(x)
        action_probs = torch.softmax(x, dim=1)
        return action_probs


The `PolicyNetwork` class defines a neural network used in reinforcement learning to model the agent's policy, which outputs the probability distribution over actions given a state. The constructor takes the number of inputs (state dimensions), the number of actions, the size of the hidden layer, and the learning rate for the optimizer. It initializes two fully connected layers: the first layer maps the input state to a hidden representation, and the second layer outputs the action probabilities. The `forward` method applies a **tanh** activation function to the hidden layer, followed by a **softmax** function to normalize the output into a valid probability distribution over actions. The network is trained using the **Adam optimizer** with the specified learning rate, optimizing the policy by adjusting its parameters during training.

# Defining the Value Network

Next, we define a value network that estimates the value of a state. This network is separate from the policy network and has its own parameters.


In [ ]:
class ValueNetwork(nn.Module):
    def __init__(self, num_inputs, hidden_size, learning_rate=3e-4):
        super(ValueNetwork, self).__init__()

        self.linear1 = nn.Linear(num_inputs, hidden_size)
        self.linear2 = nn.Linear(hidden_size, 1)
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, state):
        x = torch.tanh(self.linear1(state))
        x = self.linear2(x)
        return x


The `ValueNetwork` class defines a neural network that approximates the **value function** in reinforcement learning, which predicts the expected return (or value) of being in a given state. The constructor initializes two fully connected layers: the first layer maps the input state to a hidden representation, and the second layer outputs a single scalar value representing the state’s value. The `forward` method applies a **tanh** activation function to the hidden layer’s output and then passes it through the second layer to produce the final value estimate. The network is trained using the **Adam optimizer** with a specified learning rate to minimize the error between the predicted values and the true values during training.

# Implementing the TRPO Algorithm

Now, we will implement the TRPO algorithm. We will use the PyTorch's automatic differentiation feature to compute the gradients. The objective function and the constraint are implemented as mentioned in the task description.


In [ ]:
def trpo_step(policy_net, value_net, states, actions, rewards, masks, epsilon=0.2):
    old_action_probs = policy_net(states).gather(1, actions)

    values = value_net(states)

    advantages = rewards + masks * values - values.detach()
    new_action_probs = policy_net(states).gather(1, actions)

    ratio = new_action_probs / old_action_probs
    surrogate = ratio * advantages
    kl_divergence = old_action_probs * torch.log(old_action_probs / new_action_probs)
    loss = -surrogate + epsilon * kl_divergence
    policy_net.optimizer.zero_grad()
    loss.backward()
    policy_net.optimizer.step()

    value_net.optimizer.zero_grad()
    values.backward()
    value_net.optimizer.step()

The `trpo_step` function implements a single optimization step for **Trust Region Policy Optimization (TRPO)**. Here's a breakdown of the steps involved:

1. **Compute Old Action Probabilities**: The old action probabilities are computed by passing the states through the policy network (`policy_net`) and gathering the probabilities corresponding to the actions taken by the agent. This provides a reference to compare with the new action probabilities.

2. **Compute the Value Function**: The value function for each state is computed by passing the states through the value network (`value_net`). This estimates the expected return for each state.

3. **Compute the Advantages**: The advantage for each state is computed by subtracting the value estimate (from the value network) from the actual return (rewards plus discounted future value, weighted by the mask indicating whether the episode has ended). This advantage represents how much better the agent’s actions were compared to the value function.

4. **Compute New Action Probabilities**: The new action probabilities are computed in the same way as the old action probabilities, using the policy network with the current states.

5. **Compute the Surrogate Function**: The surrogate objective function is computed by taking the ratio of the new action probabilities to the old action probabilities, multiplied by the computed advantages. This measures how much the new policy improves over the old policy.

6. **Compute the KL Divergence**: The Kullback-Leibler (KL) divergence between the old and new action probabilities is calculated. KL divergence measures the difference between the two probability distributions and is used to ensure that the new policy does not deviate too much from the old policy.

7. **Compute the Loss**: The final loss is computed by combining the surrogate function (which the agent wants to maximize) and the KL divergence (which the agent wants to minimize). The KL divergence is scaled by a small epsilon factor to control the size of the policy update.

8. **Update the Policy Network**: The optimizer for the policy network is used to perform a gradient update based on the computed loss. The optimizer zeroes out the previous gradients, computes the gradients for the current loss, and updates the policy network’s parameters.

9. **Update the Value Network**: Similarly, the optimizer for the value network is used to perform a gradient update, this time based on the value estimates. The value network’s parameters are updated using the gradients computed from the value predictions.



# Implementing the PPO Algorithm

Next, we will implement the PPO algorithm. The PPO algorithm is similar to the TRPO algorithm, but it uses a clipped surrogate objective instead of the original surrogate objective.


In [ ]:

def ppo_step(policy_net, value_net, states, actions, rewards, masks, epsilon=0.2, beta=3.0):

    old_action_probs = policy_net(states).gather(1, actions)
    values = value_net(states)
    advantages = rewards + masks * values - values.detach()
    new_action_probs = policy_net(states).gather(1, actions)
    ratio = new_action_probs / old_action_probs
    surrogate = ratio * advantages
    kl_divergence = old_action_probs * torch.log(old_action_probs / new_action_probs)
    clipped_surrogate = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
    loss = -torch.min(surrogate, clipped_surrogate) + beta * kl_divergence

    policy_net.optimizer.zero_grad()
    loss.backward()
    policy_net.optimizer.step()

    value_net.optimizer.zero_grad()
    values.backward()
    value_net.optimizer.step()


1. **Compute Old Action Probabilities**:
   - The old action probabilities are computed by passing the states through the **policy network** (`policy_net`) and selecting the probabilities corresponding to the actions taken (using `gather(1, actions)`).

2. **Compute the Value Function**:
   - The **value function** is computed by passing the states through the **value network** (`value_net`). This function estimates the expected return for each state.

3. **Compute the Advantages**:
   - The **advantage** is computed by subtracting the current value estimate from the observed rewards, adjusted by the `masks` (which indicate whether the episode has ended). This gives an estimate of how much better the action was compared to the average action.

4. **Compute New Action Probabilities**:
   - The new action probabilities are computed in the same way as the old action probabilities by passing the states through the policy network. These new probabilities are then compared to the old probabilities to determine the ratio of the probabilities for the selected actions.

5. **Compute the Surrogate Function**:
   - The **surrogate function** is the product of the probability ratio (`new_action_probs / old_action_probs`) and the computed advantages. This function is used to evaluate the improvement of the policy.

6. **Compute the KL Divergence**:
   - The **KL divergence** is calculated to measure the difference between the old and new action probability distributions. This is used to ensure that the update to the policy does not deviate too drastically, which helps maintain stability during training.

7. **Compute the Clipped Surrogate Function**:
   - The **clipped surrogate function** is calculated by restricting the probability ratio to the range `[1 - epsilon, 1 + epsilon]` using the **clamp** function. This ensures that updates are not too large, which is crucial for stable training in PPO.

8. **Compute the Loss**:
   - The **loss** is computed as the minimum of the surrogate and clipped surrogate, which helps ensure the update is both large enough to improve performance but not too large to destabilize the training. The KL divergence is added to the loss with a weight factor `beta`, which helps control the impact of the KL divergence term on the overall loss.

9. **Update the Policy Network**:
   - The optimizer for the **policy network** (`policy_net.optimizer`) is used to update the policy parameters. The loss is backpropagated and the policy network’s parameters are updated using gradient descent.

10. **Update the Value Network**:
    - Similarly, the optimizer for the **value network** (`value_net.optimizer`) is used to update the value function parameters by performing a gradient descent step on the value estimates (calculated from the states).

### **Summary**:
This function implements a PPO step that updates both the **policy network** and the **value network**. It does so by computing a surrogate objective using the **old and new action probabilities**, **advantages**, and a **clipped version** of the objective to ensure stable updates. The **KL divergence** term is included to penalize large policy changes. Both networks are updated using gradient descent to improve the agent’s performance while maintaining stability during training.